<a href="https://colab.research.google.com/github/jayaraksha/DEEP-LEARNING-SEM-5-/blob/main/24bad044_dl_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%pip -q install "transformers>=4.45,<5" "gradio>=5,<7" "accelerate>=0.33" "safetensors>=0.4"

In [6]:
import os
import time
import json
import torch
import transformers
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gr.__version__)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Note: CPU generation is possible but slower. Enable Kaggle GPU if available.")

Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cpu
PyTorch: 2.11.0+cpu
Transformers: 4.57.6
Gradio: 6.17.3
Note: CPU generation is possible but slower. Enable Kaggle GPU if available.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer and model loaded successfully!")
print("Number of parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tokenizer and model loaded successfully!
Number of parameters: 494,032,768


In [4]:
sample_prompt = "Explain artificial intelligence in simple words."
sample_tokens = tokenizer(sample_prompt, return_tensors="pt")
print("Prompt:", sample_prompt)
print("Token IDs:", sample_tokens["input_ids"][0].tolist())
print("Tokens:", tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"][0]))
print("Token count:", sample_tokens["input_ids"].shape[1])

Prompt: Explain artificial intelligence in simple words.
Token IDs: [840, 20772, 20443, 11229, 304, 4285, 4244, 13]
Tokens: ['Ex', 'plain', 'Ġartificial', 'Ġintelligence', 'Ġin', 'Ġsimple', 'Ġwords', '.']
Token count: 8


In [7]:
SYSTEM_PROMPT = (
    "You are a helpful educational chatbot. Give accurate, clear, concise answers. "
    "If unsure, say you are uncertain rather than inventing facts."
)

def generate_response(message, history=None, temperature=0.7, max_new_tokens=128):
    """Generate a response using the prompt and recent user/assistant conversation."""
    message = str(message).strip()
    if not message:
        return "Please type a question."

    history = history or []
    valid_history = [
        {"role": item["role"], "content": item["content"]}
        for item in history[-6:]
        if isinstance(item, dict)
        and item.get("role") in ("user", "assistant")
        and isinstance(item.get("content"), str)
    ]
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + valid_history
        + [{"role": "user", "content": message}]
    )
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=1536,
    ).to(DEVICE)

    use_sampling = float(temperature) > 0
    kwargs = {
        "max_new_tokens": int(max_new_tokens),
        "do_sample": use_sampling,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if use_sampling:
        kwargs.update(temperature=float(temperature), top_p=0.9)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **kwargs)
    new_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [8]:
question = "What is deep learning? Answer in two sentences."
for temp in (0.0, 0.7, 1.0):
    start = time.perf_counter()
    answer = generate_response(question, temperature=temp, max_new_tokens=80)
    print(f"\nTemperature = {temp} | Time = {time.perf_counter()-start:.2f}s")
    print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Temperature = 0.0 | Time = 25.63s
Deep learning is a subset of machine learning that involves training artificial neural networks to perform tasks such as image recognition, speech processing, and natural language understanding by allowing the network to learn complex patterns from large datasets.

Temperature = 0.7 | Time = 10.34s
Deep learning is an area of artificial intelligence that involves the use of neural networks to learn and make decisions from large datasets.

Temperature = 1.0 | Time = 20.16s
Deep learning is a subset of machine learning that focuses on the use of artificial neural networks to perform tasks such as image and speech recognition, natural language processing, and predictive analytics. It allows computers to learn from data patterns and make predictions or decisions based on these learned patterns.


part b


In [9]:
chat_history = []
questions = [
    "My favorite subject is machine learning.",
    "What is my favorite subject?",
]
for q in questions:
    a = generate_response(q, chat_history, temperature=0.3, max_new_tokens=70)
    chat_history.extend([
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ])
    print("USER:", q)
    print("BOT:", a, "\n")

USER: My favorite subject is machine learning.
BOT: That's great to hear! Machine learning is indeed one of the most fascinating and rapidly evolving fields in technology today. It involves teaching computers to learn from data without being explicitly programmed to do so. This makes it incredibly useful for tasks like image recognition, natural language processing, predictive analytics, and more.

Machine learning has many applications across various industries, including 

USER: What is my favorite subject?
BOT: As an AI model, I don't have personal preferences or favorites. However, I can provide information on different subjects if that would be helpful! What specific topic would you like to know about? 



part c

In [10]:
def chatbot_reply(message, history, temperature, max_tokens):
    return generate_response(message, history, temperature, max_tokens)

chat_app = gr.ChatInterface(
    fn=chatbot_reply,
    title="🤖 Transformer LLM Chatbot",
    description="Chat with a pre-trained Qwen2.5-0.5B model. Built using Hugging Face Transformers and Gradio.",
    additional_inputs=[
        gr.Slider(minimum=0, maximum=1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(minimum=32, maximum=256, value=128, step=32, label="Maximum new tokens"),
    ],
    examples=[
        ["What is artificial intelligence?", 0.7, 128],
        ["Explain the Transformer architecture in simple words.", 0.5, 128],
        ["Give three applications of machine learning.", 0.7, 128],
    ],
)

# share=False avoids requiring an external public Gradio tunnel.
# Set share=True ONLY if you intentionally want a temporary public link and
# Kaggle networking allows it.
chat_app.launch(
    share=True,inline=False,debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9aed5b27602d4ea5f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9aed5b27602d4ea5f0.gradio.live


part d

In [11]:
test_prompts = [
    "What is machine learning?",
    "Give two differences between AI and deep learning.",
    "Write a short motivational message for students.",
]
test_results = []
for prompt in test_prompts:
    start = time.perf_counter()
    response = generate_response(prompt, temperature=0.4, max_new_tokens=100)
    latency = round(time.perf_counter() - start, 2)
    test_results.append({"prompt": prompt, "response": response, "latency_seconds": latency})
    print("PROMPT:", prompt)
    print("RESPONSE:", response)
    print("LATENCY:", latency, "seconds\n")
print("Average response latency:", round(sum(r["latency_seconds"] for r in test_results)/len(test_results), 2), "seconds")

PROMPT: What is machine learning?
RESPONSE: Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from data and improve their performance over time without being explicitly programmed. In other words, it's about making machines "learn" or adapt based on new information.

Key aspects of machine learning include:

1. **Data**: The input for a machine learning model comes in the form of raw data.
2. **Model**: This is a mathematical formula or algorithm designed to make predictions or decisions based
LATENCY: 24.39 seconds

PROMPT: Give two differences between AI and deep learning.
RESPONSE: Certainly! Here are the key differences between Artificial Intelligence (AI) and Deep Learning:

1. **Definition**:
   - **Artificial Intelligence**: This encompasses all forms of intelligence that are not just machine-like but also human-like. It includes tasks such as perception, reasoning, learning, 